##Dataset overview

- We will be using a credit scoring dataset.
- It includes variables such as `income`, `loan_amount`, `term`, `credit_history`, and `defaulted`.

## Install External Libraries

In [0]:
!pip install altair

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Create Catalog/Schema/Volume/Folder

In [0]:
%sql
create catalog if not exists credit_scoring;

In [0]:
%sql
create schema if not exists credit_scoring.bronze;
create schema if not exists credit_scoring.models;

In [0]:
%sql
create volume if not exists credit_scoring.bronze.bronze_vol;

In [0]:
import os

os.makedirs("/Volumes/credit_scoring/bronze/bronze_vol/files", exist_ok=True)

## Load Github Dataset Into Volume/Folder


In [0]:
import requests

# response = requests.get('https://raw.githubusercontent.com/juanjopc/Proyecto-Databricks-Machine-Learning/main/bank_loans.csv')
response = requests.get('https://raw.githubusercontent.com/juanjopc/Proyecto-Databricks-Machine-Learning/refs/heads/main/bank_loans.csv')

with open('/Volumes/credit_scoring/bronze/bronze_vol/files/bank_loans.csv', 'wb') as file:
    file.write(response.content)

## Load Dataset into Pandas Dataframe

In [0]:
import pandas as pd

df = pd.read_csv('/Volumes/credit_scoring/bronze/bronze_vol/files/bank_loans.csv')
display(df.head(5))

income,loan_amount,term,credit_history,defaulted
60940.85475420218,17402.715470744486,60,1.0,1
49511.24257891868,6664.626122778236,36,0.0,0
63658.39368581247,17985.281393286743,60,0.0,1
79414.53741534446,21001.17376965355,36,1.0,0
47785.23925497996,4037.586144571267,36,1.0,0


## EDA

In [0]:
df.shape

(1248, 5)

In [0]:
df.dtypes

income            float64
loan_amount       float64
term                int64
credit_history    float64
defaulted           int64
dtype: object

In [0]:
df.isnull().sum()

income            120
loan_amount       122
term                0
credit_history    112
defaulted           0
dtype: int64

Montos:
Si no sé cuánto gana la persona ni cuanto fue el prestamo que solicitó, asumiré que fue lo mismo que el individuo del centro de la población (mediana).

In [0]:
df.fillna({'income':df['income'].median()}, inplace=True)
df.fillna({'loan_amount':df['loan_amount'].median()}, inplace=True)

Categorias:
Si no se si la persona tuvo o no tuvo historial crediticio, apostaré por el escenario mas probable (moda).

In [0]:
df.fillna({'credit_history':df['credit_history'].mode()[0]}, inplace=True)
df['credit_history'] = df['credit_history'].astype(int)

In [0]:
df.isnull().sum()

income            0
loan_amount       0
term              0
credit_history    0
defaulted         0
dtype: int64

In [0]:
df.describe()

,income,loan_amount,term,credit_history,defaulted
count,1248.000000,1248.000000,1248.000000,1248.000000,1248.000000
mean,52500.945298,16167.787505,48.000000,0.771635,0.414263
std,16967.161028,5541.521139,12.004811,0.419948,0.492792
min,8000.000000,1000.000000,36.000000,0.000000,0.000000
25%,41673.912889,12772.722413,36.000000,1.000000,0.000000
50%,52462.564119,16052.247335,48.000000,1.000000,0.000000
75%,62987.634437,19610.211517,60.000000,1.000000,1.000000
max,121349.166832,35158.645407,60.000000,1.000000,1.000000


## Step 4: Data Visualization

In [0]:
import altair as alt

In [0]:
chart = alt.Chart(df).mark_bar(tooltip=True).encode(
        x=alt.X("income:Q").bin(maxbins=30).title("Income"),
        y=alt.Y('count()').title('Count')
)

text = chart.mark_text(dy=-5).encode(
    text=alt.Text('count()')
)

final_chart = (chart + text).properties(
    width=450,
    height=300
)

#############################################

chart2 = alt.Chart(df).mark_bar(tooltip=True).encode(
        x=alt.X("loan_amount:Q").bin(maxbins=30).title("Loan Amount"),
        y=alt.Y('count()').title('Count')
)

text2 = chart2.mark_text(dy=-5).encode(
    text=alt.Text('count()')
)

final_chart2 = (chart2 + text2).properties(
    width=450,
    height=300
)

#############################################

final_chart | final_chart2

alt.HConcatChart(...)

In [0]:
chart = alt.Chart(df).mark_bar(tooltip=True).encode(
    x = alt.X("term:N").title("Loan Term (Months)"),
    y = alt.Y("count()").title("Count"),
).properties(
    width=400
)

chart

alt.Chart(...)

In [0]:
chart = alt.Chart(df).mark_bar().encode(
    x = alt.X("credit_history:N").title("Credit History"),
    y = alt.Y("count()").title("Count"),
    color = alt.Color("defaulted:N").title("Defaulted"),
    xOffset = alt.XOffset("defaulted:N"),
    tooltip = ["credit_history:N", "defaulted:N", "count()"]
).properties(
    width = 400
)

chart

alt.Chart(...)

In [0]:
corr_matrix = df.corr(numeric_only=True).reset_index()
display(corr_matrix)

index,income,loan_amount,term,credit_history,defaulted
income,1.0,0.012194661751961149,0.00364646473971941,-0.05572977138083541,-0.0924059884691135
loan_amount,0.012194661751961149,1.0,-0.031412117862204623,0.08665156886165788,0.12492198672394521
term,0.00364646473971941,-0.031412117862204623,1.0,0.017179358231248876,0.00487997160168428
credit_history,-0.05572977138083541,0.08665156886165788,0.017179358231248876,1.0,-0.25550046543877825
defaulted,-0.0924059884691135,0.12492198672394521,0.00487997160168428,-0.25550046543877825,1.0


In [0]:
corr_df = corr_matrix.melt("index", var_name="variable", value_name="correlation")
display(corr_df)

index,variable,correlation
income,income,1.0
loan_amount,income,0.012194661751961149
term,income,0.00364646473971941
credit_history,income,-0.05572977138083541
defaulted,income,-0.0924059884691135
income,loan_amount,0.012194661751961149
loan_amount,loan_amount,1.0
term,loan_amount,-0.031412117862204623
credit_history,loan_amount,0.08665156886165788
defaulted,loan_amount,0.12492198672394521


In [0]:
chart = alt.Chart(corr_df).mark_rect().encode(
    x=alt.X("index:O"),
    y=alt.Y("variable:O"),
    color=alt.Color("correlation:Q").scale(scheme="redblue", domain=[-1, 1])
)

text = chart.mark_text().encode(
    text = alt.Text("correlation:Q", format=".2f"),
    color = alt.condition(
        abs(alt.datum.correlation) > 0.5,
        alt.value('white'),
        alt.value("black")
    )
)

final_chart = (chart + text).properties(
    title="Feature Correlation Heatmap",
    width=600,
    height=300
)

final_chart

alt.LayerChart(...)

## Step 5: Feature Engineering

In [0]:
df['term_binary'] = df['term'].apply(lambda x: 1 if x == 60 else 0)

Se aplica una transformación logarítmica para reducir la distancia entre los valores extremadamente altos de los altos.

In [0]:

import numpy as np
df['log_income'] = np.log1p(df['income'])
df['log_loan_amount'] = np.log1p(df['loan_amount'])

In [0]:
list(df.columns)

['income',
 'loan_amount',
 'term',
 'credit_history',
 'defaulted',
 'term_binary',
 'log_income',
 'log_loan_amount']

In [0]:
features = ['log_income', 'log_loan_amount', 'term_binary', 'credit_history']
target = 'defaulted'

## Step 7: Model Training

In [0]:
import mlflow

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report

In [0]:
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
exp = mlflow.get_experiment_by_name("/Users/uipython123@gmail.com/credit_scoring/experimento_1")

if exp is not None:
    mlflow.delete_experiment(exp.experiment_id)

In [0]:
mlflow.set_experiment("/Users/uipython123@gmail.com/credit_scoring/experimento_1")
mlflow.autolog(log_input_examples=True)

2026/03/25 12:58:40 INFO mlflow.tracking.fluent: Experiment with name '/Users/uipython123@gmail.com/credit_scoring/experimento_1' does not exist. Creating a new experiment.
2026/03/25 12:58:41 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/25 12:58:41 WARNING mlflow.utils.autologging_utils: MLflow spark autologging is known to be compatible with 3.1.2 <= pyspark <= 3.5.5, but the installed version is 4.0.0. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/03/25 12:58:41 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: MLflow Spark dataset autologging is not supported on Databricks shared clusters or Databricks serverless clusters.
2026/03/25 12:58:41 WARNING mlflow.utils.autologging_utils: MLflow pyspark.ml autologging is known to be compatible with 3.1.2 <= pyspark <= 3.5.5, but the installed version is 4.0.0. If you encounte

para poder hacer el machine learning colocamos el promedio como 0 en las columnas numericas

In [0]:
preprocessor = ColumnTransformer(
    transformers=[('num', StandardScaler(), ['log_income', 'log_loan_amount'])],
    remainder='passthrough'
)

In [0]:
with mlflow.start_run(run_name="Logistic_Regression"):
    lr_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    lr_pipeline.fit(X_train, y_train)
    y_pred_lr = lr_pipeline.predict(X_test)
    print(classification_report(y_test, y_pred_lr))
    report_dict = classification_report(y_test, y_pred_lr, output_dict=True)
        
    #Anotar
    mlflow.log_metrics({
        "test_accuracy": report_dict["accuracy"],
        "test_precision": report_dict["macro avg"]["precision"],
        "test_recall": report_dict["macro avg"]["recall"],
        "test_f1_score": report_dict["macro avg"]["f1-score"]
    })

2026/03/25 12:58:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/25 12:58:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/sk

              precision    recall  f1-score   support

           0       0.66      0.87      0.75       143
           1       0.69      0.40      0.51       107

    accuracy                           0.67       250
   macro avg       0.68      0.63      0.63       250
weighted avg       0.67      0.67      0.65       250



In [0]:
with mlflow.start_run(run_name="Decision_Tree"):
    dt_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
    ])
    dt_pipeline.fit(X_train, y_train)
    y_pred_dt = dt_pipeline.predict(X_test)
    print(classification_report(y_test, y_pred_dt))
    report_dict = classification_report(y_test, y_pred_dt, output_dict=True)
    
    # Anotar Métricas
    mlflow.log_metrics({
        "test_accuracy": report_dict["accuracy"],
        "test_precision": report_dict["macro avg"]["precision"],
        "test_recall": report_dict["macro avg"]["recall"],
        "test_f1_score": report_dict["macro avg"]["f1-score"]
    })

2026/03/25 12:58:50 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/25 12:58:51 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/sk

              precision    recall  f1-score   support

           0       0.65      0.83      0.73       143
           1       0.64      0.41      0.50       107

    accuracy                           0.65       250
   macro avg       0.64      0.62      0.61       250
weighted avg       0.65      0.65      0.63       250



In [0]:
with mlflow.start_run(run_name="Random_Forest") as run:
    rf_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    rf_pipeline.fit(X_train, y_train)
    y_pred_rf = rf_pipeline.predict(X_test)
    print(classification_report(y_test, y_pred_rf))
    report_dict = classification_report(y_test, y_pred_rf, output_dict=True)
    
    # Anotar Métricas
    mlflow.log_metrics({
        "test_accuracy": report_dict["accuracy"],
        "test_precision": report_dict["macro avg"]["precision"],
        "test_recall": report_dict["macro avg"]["recall"],
        "test_f1_score": report_dict["macro avg"]["f1-score"]
    })

2026/03/25 12:58:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/25 12:58:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/sk

              precision    recall  f1-score   support

           0       0.65      0.70      0.67       143
           1       0.55      0.49      0.51       107

    accuracy                           0.61       250
   macro avg       0.60      0.59      0.59       250
weighted avg       0.60      0.61      0.60       250

